# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [2]:
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [ ]:
get_food_calories('banana')

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """)

In [ ]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

Enforce tools use:

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

## EVS Assistant

Create a lookup tool for four environmental studies concepts:

In [4]:
@function_tool
def get_evs_concept(concept: str) -> str:
    """Look up a concise, student-friendly definition of an EVS concept.

    Args:
        concept: Photosynthesis, ozone, global warming, or the greenhouse effect.

    Returns:
        The concept's definition, or the supported concepts if no match is found.
    """
    concept_key = " ".join(
        concept.lower().replace("-", " ").replace("?", "").replace(".", "").split()
    )
    for prefix in ("what is ", "define ", "explain ", "tell me about "):
        if concept_key.startswith(prefix):
            concept_key = concept_key.removeprefix(prefix)
            break
    if concept_key.endswith(" definition"):
        concept_key = concept_key.removesuffix(" definition")

    concept_aliases = {
        "photosynthesis": "Photosynthesis",
        "photosyntensis": "Photosynthesis",
        "ozone": "Ozone",
        "ozon": "Ozone",
        "o3": "Ozone",
        "global warming": "Global warming",
        "greenhouse effect": "Greenhouse effect",
        "green house effect": "Greenhouse effect",
    }
    definitions = {
        "Photosynthesis": (
            "Photosynthesis is the process by which green plants use sunlight, "
            "water, and carbon dioxide to make food (glucose), releasing oxygen."
        ),
        "Ozone": (
            "Ozone (O3) is a gas made of three oxygen atoms. In the stratosphere, "
            "the ozone layer absorbs much of the Sun's harmful ultraviolet radiation."
        ),
        "Global warming": (
            "Global warming is the long-term rise in Earth's average temperature, "
            "mainly caused today by human emissions of heat-trapping greenhouse gases."
        ),
        "Greenhouse effect": (
            "The greenhouse effect is the natural process in which gases in the "
            "atmosphere trap some heat and warm Earth. Human activities are "
            "strengthening this effect and contributing to global warming."
        ),
    }

    matched_concept = concept_aliases.get(concept_key)
    if matched_concept is None:
        return (
            f"I don't have a definition for '{concept}'. Try photosynthesis, "
            "ozone, global warming, or greenhouse effect."
        )
    return f"{matched_concept}: {definitions[matched_concept]}"

In [5]:
evs_agent = Agent(
    name="EVS Assistant",
    instructions="""
    You are a helpful Environmental Studies (EVS) assistant for students.
    Use get_evs_concept to look up the four supported concepts and explain
    the result in clear, concise language.
    """,
    tools=[get_evs_concept],
    model_settings=ModelSettings(tool_choice="get_evs_concept"),
)

with trace("EVS Assistant with concept lookup"):
    result = await Runner.run(
        evs_agent, "What is the greenhouse effect?"
    )
    print(result.final_output)

Definition:
- The greenhouse effect is the natural process where certain gases in Earth’s atmosphere trap some of the Sun’s heat, warming the planet to a livable level. Human activities are increasing this effect, which contributes to additional warming known as global warming.

Key points:
- It’s a normal, essential process that keeps Earth warm enough for life.
- Greenhouse gases include carbon dioxide (CO2), methane (CH4), nitrous oxide (N2O), and fluorinated gases.
- Human activities—burning fossil fuels, deforestation, and certain agricultural practices—add more of these gases, trapping more heat and changing climate patterns.

Simple analogy: It’s like a blanket around the Earth that keeps heat from escaping. More blankets (more greenhouse gases) mean a warmer planet.
